In [ ]:
# CSV 데이터를 표 형태로 처리하기 위해 pandas를 가져옵니다.
import pandas as pd

# Hugging Face 비공개 데이터셋에서 파일을 받는 함수를 가져옵니다.
from huggingface_hub import hf_hub_download

# 프로젝트 루트의 .env에서 인증 토큰을 읽는 공통 모듈을 가져옵니다.
from common import secrets
from supply.model_data import holdout_safe_frame

# 프로젝트 루트의 .env에서 본인의 Hugging Face 토큰을 읽습니다.
token, token_source = secrets.load_key(
    ("HUGGINGFACE_ACCESS_TOKEN",),
)

# 토큰이 비어 있다면 잘못된 인증 요청을 보내지 않고 즉시 알려줍니다.
if not token:
    raise RuntimeError(
        "HUGGINGFACE_ACCESS_TOKEN을 찾지 못했습니다. "
        "프로젝트 루트의 .env 파일을 확인해주세요."
    )

# Hugging Face 토큰은 hf_로 시작하는 ASCII 문자열이어야 합니다.
#
# 이 검사를 통해 .env의 한글 주석이나 예시 문장을
# 토큰으로 잘못 읽는 문제를 다운로드 전에 발견합니다.
if not token.startswith("hf_") or not token.isascii():
    raise RuntimeError(
        "Hugging Face 토큰 형식이 올바르지 않습니다. "
        ".env의 HUGGINGFACE_ACCESS_TOKEN 줄에는 실제 토큰만 입력해주세요."
    )

# 팀의 비공개 Hugging Face 데이터셋에서
# 준영님에게 제공된 KOSPI200 피처·라벨 CSV 파일을 받습니다.
#
# 파일은 프로젝트 폴더가 아니라 Hugging Face 캐시에 저장되므로
# CSV 원본이 실수로 GitHub에 커밋되지 않습니다.
data_path = hf_hub_download(
    repo_id="qurious-quant/alphastack-krx-dev",
    filename="small/features_labels_kospi200_dev.csv",
    repo_type="dataset",
    token=token,
)

# 기준일이 숫자로 변형되지 않도록 bas_dd를 문자열로 지정해 CSV를 읽습니다.
df = pd.read_csv(
    data_path,
    dtype={"bas_dd": "string"},
)

# 모델에는 봉인 시작일 이전의 개발구간만 전달합니다. 원본 파일에 홀드아웃 행이
# 섞여 있으면 경계 직전 5거래일도 제거해 미래 5거래일 라벨의 교차를 막습니다.
df = holdout_safe_frame(df, label_horizon=5)
holdout_filter = df.attrs["holdout_filter"]
print("홀드아웃 차단:", holdout_filter)

# 토큰 전체는 출력하지 않고 어느 파일에서 읽었는지만 확인합니다.
print("토큰 출처:", token_source)

# 데이터의 행 개수와 열 개수를 확인합니다.
print("전체 데이터 크기:", df.shape)

# 모델 입력 X에 넣지 않을 열을 지정합니다.
#
# 날짜·지수 이름은 모델이 학습할 수치 피처가 아니므로 제외합니다.
# 시가·고가·저가·종가 등의 원본값도 현재 기준 모델 피처에서 제외합니다.
# fwd_return_5d와 label은 미래 정보를 담고 있으므로 반드시 X에서 제외합니다.
NOT_FEATURE = {
    "bas_dd",
    "date",
    "index_name",
    "index_class",
    "open",
    "high",
    "low",
    "close",
    "change",
    "change_rate",
    "volume",
    "value",
    "market_cap",
    "fwd_return_5d",
    "label",
}

# 모델 학습에 반드시 필요한 기준일과 정답 열이 존재하는지 확인합니다.
REQUIRED_COLUMNS = {
    "bas_dd",
    "label",
}

# 필수 열 중 데이터에 없는 열을 찾습니다.
missing_columns = REQUIRED_COLUMNS - set(df.columns)

# 필수 열이 없다면 이후 학습을 진행하지 않고 누락된 열을 알려줍니다.
if missing_columns:
    raise ValueError(
        f"필수 열이 없습니다: {sorted(missing_columns)}"
    )

# 제외 대상이 아닌 나머지 열을 모델 입력 피처로 선택합니다.
#
# CSV에 저장된 기존 열 순서를 그대로 유지합니다.
FEATURE_COLUMNS = [
    column
    for column in df.columns
    if column not in NOT_FEATURE
]

# 선택한 피처들로 모델 입력값 X를 만듭니다.
X = df.loc[:, FEATURE_COLUMNS].copy()

# 미래 5거래일 방향 라벨을 모델의 정답 y로 만듭니다.
#
# 하락=-1, 중립=0, 상승=1입니다.
y = df["label"].copy()

# 선택된 피처 개수와 이름을 확인합니다.
print("X 크기:", X.shape)
print("y 크기:", y.shape)
print("피처 개수:", len(FEATURE_COLUMNS))
print("피처 목록:", FEATURE_COLUMNS)

# X에 학습할 행이나 피처가 하나도 없는지 확인합니다.
if X.empty:
    raise ValueError("모델에 사용할 X 데이터가 비어 있습니다.")

# 원본 라벨에 결측값이 있는지 확인합니다.
missing_label_count = int(y.isna().sum())

if missing_label_count > 0:
    raise ValueError(
        f"y에 결측 라벨이 {missing_label_count}개 있습니다."
    )

# CSV의 한글 라벨 앞뒤에 불필요한 공백이 있을 가능성을 제거합니다.
y_text = y.astype("string").str.strip()

# Hugging Face 데이터셋에서 사용하는 한글 라벨을 지정합니다.
EXPECTED_TEXT_LABELS = {
    "하락",
    "중립",
    "상승",
}

# 실제 데이터에 들어 있는 라벨을 확인합니다.
observed_text_labels = set(
    y_text.dropna().unique().tolist()
)

# 정해진 세 라벨에 포함되지 않는 값이 있는지 확인합니다.
unexpected_labels = observed_text_labels - EXPECTED_TEXT_LABELS

if unexpected_labels:
    raise ValueError(
        "알 수 없는 라벨이 포함되어 있습니다: "
        f"{sorted(unexpected_labels)}"
    )

# CSV의 한글 라벨을 모델이 사용할 숫자 라벨로 변환합니다.
#
# 하락=-1
# 중립=0
# 상승=1
LABEL_TO_NUMBER = {
    "하락": -1,
    "중립": 0,
    "상승": 1,
}

# 한글 라벨을 숫자 라벨로 변환합니다.
y = y_text.map(LABEL_TO_NUMBER)

# 변환되지 않은 라벨이 남아 있는지 확인합니다.
#
# 한글 오타나 예상하지 못한 값이 있다면 map() 결과가 결측값이 됩니다.
unmapped_label_count = int(y.isna().sum())

if unmapped_label_count > 0:
    raise ValueError(
        f"숫자로 변환하지 못한 라벨이 {unmapped_label_count}개 있습니다."
    )

# 모든 라벨이 정상적으로 변환된 뒤 정수형으로 변경합니다.
y = y.astype(int)

# 변환된 숫자 라벨의 종류를 확인합니다.
observed_numeric_labels = sorted(y.unique().tolist())

# 프로젝트가 사용하는 세 클래스와 일치하는지 확인합니다.
if observed_numeric_labels != [-1, 0, 1]:
    raise ValueError(
        "숫자 라벨은 하락=-1, 중립=0, 상승=1이어야 합니다. "
        f"현재 라벨: {observed_numeric_labels}"
    )

# X에 문자열처럼 모델이 바로 학습할 수 없는 열이 있는지 확인합니다.
non_numeric_features = [
    column
    for column in FEATURE_COLUMNS
    if not pd.api.types.is_numeric_dtype(X[column])
]

if non_numeric_features:
    raise TypeError(
        "숫자가 아닌 피처가 포함되어 있습니다: "
        f"{non_numeric_features}"
    )

# 전체 피처에 포함된 결측값 개수를 계산합니다.
missing_feature_count = int(X.isna().sum().sum())

# 라벨 분포는 사람이 읽기 쉬운 한글 라벨을 기준으로 계산합니다.
#
# reindex를 사용해 항상 하락·중립·상승 순서로 출력되도록 합니다.
label_distribution = (
    y_text.value_counts(normalize=True)
    .reindex(["하락", "중립", "상승"], fill_value=0)
    .mul(100)
    .round(2)
)

# 원본 행은 출력하지 않고 학습에 필요한 요약 정보만 확인합니다.
print("데이터 시작일:", df["bas_dd"].min())
print("데이터 종료일:", df["bas_dd"].max())
print("X 크기:", X.shape)
print("y 크기:", y.shape)
print("X 전체 결측값:", missing_feature_count)
print("숫자 라벨:", observed_numeric_labels)
print("라벨 변환:", LABEL_TO_NUMBER)
print("라벨 분포(%):")
print(label_distribution)

# RandomForest — 조합C + Daily_Return

## 실험 목적

기존 조합 C에 `daily_return` 수익률 피처를 추가해 같은 12개
워크포워드 폴드에서 성능 변화를 확인한다. 수익률은 현재와 과거 종가만 사용하며,
모델의 클래스 가중치는 기존 조합에서 선택한 **balanced** 설정을 유지한다.


In [2]:
# 프로젝트 루트를 sys.path에 넣은 뒤 내부 모듈을 가져와야 합니다.
# ruff: noqa: E402

# 파일 경로와 프로젝트 모듈 연결에 필요한 기능을 가져옵니다.
import sys
from pathlib import Path

# 현재 위치에서 프로젝트 루트를 찾습니다.
project_root = Path.cwd().resolve()

while (
    project_root != project_root.parent
    and not (project_root / "models").is_dir()
):
    project_root = project_root.parent

if not (project_root / "models").is_dir():
    raise RuntimeError(
        "프로젝트 루트의 models 폴더를 찾지 못했습니다."
    )

# 프로젝트 모듈을 import할 수 있도록 경로를 추가합니다.
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 데이터 처리에 필요한 라이브러리를 가져옵니다.
import numpy as np
import pandas as pd

# Hugging Face에서 데이터 파일을 받는 함수를 가져옵니다.
from huggingface_hub import hf_hub_download

# 모델 평가에 필요한 함수를 가져옵니다.
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

# 프로젝트의 인증키 로딩 함수를 가져옵니다.
from common import secrets

# 프로젝트에서 만든 기준선 예측 함수를 가져옵니다.
from evaluation.baseline import (
    always_up,
    majority_class,
)

# 방향 적중률과 워크포워드 분할 함수를 가져옵니다.
from evaluation.metrics import hit_rate
from evaluation.walk_forward import expanding_splits

# RandomForest 모델을 가져옵니다.
from models.random_forest import build_random_forest_baseline

print("프로젝트 루트:", project_root)

프로젝트 루트: C:\Users\Administrator\Alpha_Stack


In [3]:
# 실험 이름과 조합 C의 피처 이름을 지정합니다.
EXPERIMENT_NAME = "조합 C"

FEATURE_COMBINATION = [
    "sma_gap_5_20",
    "macd_hist_ratio",
    "rsi_14",
    "bb_position",
    "hv_20",
    "vol_ratio_20",
]

# 워크포워드 실험 설정값을 지정합니다.
N_FOLDS = 12
MIN_TRAIN_SIZE = 750
VALID_SIZE = 60
LABEL_HORIZON = 5

# 결과표에서 사용할 클래스 순서와 이름을 지정합니다.
CLASS_LABELS = [-1, 0, 1]

CLASS_NAMES = {
    -1: "하락",
    0: "중립",
    1: "상승",
}

# 조합 C 계산에 필요한 기존 열을 실수형으로 변환합니다.
close = pd.to_numeric(
    df["close"],
    errors="raise",
).astype(float)

sma_5 = pd.to_numeric(
    df["sma_5"],
    errors="raise",
).astype(float)

sma_20 = pd.to_numeric(
    df["sma_20"],
    errors="raise",
).astype(float)

macd_hist = pd.to_numeric(
    df["macd_hist"],
    errors="raise",
).astype(float)

rsi_14 = pd.to_numeric(
    df["rsi_14"],
    errors="raise",
).astype(float)

bb_upper = pd.to_numeric(
    df["bb_upper"],
    errors="raise",
).astype(float)

bb_lower = pd.to_numeric(
    df["bb_lower"],
    errors="raise",
).astype(float)

hv_20 = pd.to_numeric(
    df["hv_20"],
    errors="raise",
).astype(float)

vol_ratio_20 = pd.to_numeric(
    df["vol_ratio_20"],
    errors="raise",
).astype(float)

# 0으로 나누는 문제가 있는지 확인합니다.
if (sma_20 == 0).any():
    raise RuntimeError(
        "sma_20에 0이 있어 이동평균 차이를 계산할 수 없습니다."
    )

if (close == 0).any():
    raise RuntimeError(
        "close에 0이 있어 MACD 비율을 계산할 수 없습니다."
    )

# 볼린저밴드 상단과 하단의 차이를 계산합니다.
bb_range = bb_upper - bb_lower

if (bb_range == 0).any():
    raise RuntimeError(
        "볼린저밴드 폭이 0인 행이 있습니다."
    )

# 5일 이동평균과 20일 이동평균의 상대적인 차이를 계산합니다.
#
# 양수이면 최근 상승 추세,
# 음수이면 최근 하락 추세일 가능성을 나타냅니다.
sma_gap_5_20 = (
    sma_5 / sma_20
) - 1.0

# MACD 히스토그램을 현재 종가로 나눠 정규화합니다.
#
# 양수이면 상승 모멘텀,
# 음수이면 하락 모멘텀일 가능성을 나타냅니다.
macd_hist_ratio = (
    macd_hist / close
)

# 현재 가격이 볼린저밴드 안에서 어느 위치인지 계산합니다.
#
# 0에 가까우면 하단,
# 0.5에 가까우면 중심,
# 1에 가까우면 상단에 있는 상태입니다.
bb_position = (
    (close - bb_lower) / bb_range
)

# 조합 C의 여섯 가지 피처로 X를 만듭니다.
X_selected = pd.DataFrame(
    {
        "sma_gap_5_20": sma_gap_5_20,
        "macd_hist_ratio": macd_hist_ratio,
        "rsi_14": rsi_14,
        "bb_position": bb_position,
        "hv_20": hv_20,
        "vol_ratio_20": vol_ratio_20,
    },
    index=df.index,
)

# 무한대 값을 결측값으로 변경해 검사할 수 있게 합니다.
X_selected = X_selected.replace(
    [np.inf, -np.inf],
    np.nan,
)

# 위쪽 데이터 준비 과정에서 만든 y를 정수 배열로 준비합니다.
# 현재와 과거 종가만 사용하는 수익률 피처를 추가합니다.
return_close = pd.to_numeric(df["close"], errors="raise").astype(float)
X_selected["daily_return"] = return_close / return_close.shift(1) - 1.0
FEATURE_COMBINATION = [*FEATURE_COMBINATION, "daily_return"]

y_numeric = y.astype(int).to_numpy()

# 데이터가 날짜 오름차순인지 확인합니다.
if not df["bas_dd"].is_monotonic_increasing:
    raise RuntimeError(
        "데이터가 날짜 오름차순으로 정렬되어 있지 않습니다."
    )

# X와 y의 데이터 수가 같은지 확인합니다.
if len(X_selected) != len(y_numeric):
    raise RuntimeError(
        "X와 y의 데이터 개수가 서로 다릅니다."
    )

# 조합 C에 결측값이 있는지 확인합니다.
missing_value_count = int(
    X_selected.isna().sum().sum()
)

expected_warmup_missing = 1
if missing_value_count != expected_warmup_missing:
    raise RuntimeError(
        f"수익률 워밍업 외 결측값이 있습니다: {missing_value_count}개"
    )

print("실험 모델: RandomForest")
print("피처 조합:", EXPERIMENT_NAME)
print("사용 피처:", FEATURE_COMBINATION)
print("X 크기:", X_selected.shape)
print("y 크기:", y_numeric.shape)
print("X 전체 결측값:", missing_value_count)
print("숫자 라벨:", sorted(np.unique(y_numeric).tolist()))

display(X_selected.describe().round(6))

실험 모델: RandomForest
피처 조합: 조합 C
사용 피처: ['sma_gap_5_20', 'macd_hist_ratio', 'rsi_14', 'bb_position', 'hv_20', 'vol_ratio_20', 'daily_return']
X 크기: (2815, 7)
y 크기: (2815,)
X 전체 결측값: 1
숫자 라벨: [-1, 0, 1]


,sma_gap_5_20,macd_hist_ratio,rsi_14,bb_position,hv_20,vol_ratio_20,daily_return
count,2815.000000,2815.000000,2815.000000,2815.000000,2815.000000,2815.000000,2814.000000
mean,0.001824,-0.000032,52.440157,0.542582,0.009732,1.005557,0.000272
std,0.021021,0.003831,12.094499,0.324928,0.004925,0.222175,0.010913
min,-0.162245,-0.032638,12.760332,-0.294551,0.003077,0.501099,-0.076681
25%,-0.009042,-0.002023,44.156156,0.286998,0.006924,0.855454,-0.004863
50%,0.002958,-0.000048,52.776217,0.585355,0.008491,0.968475,0.000492
75%,0.013649,0.002114,61.049809,0.806872,0.010838,1.114158,0.005936
max,0.089045,0.018248,85.810765,1.326310,0.044466,2.394818,0.091495


In [4]:
# 확장형 워크포워드 학습·검증 구간을 만듭니다.
#
# horizon=60은 폴드별 검증 데이터 개수입니다.
# label_horizon=5는 y가 미래 5거래일을 사용한다는 의미입니다.
# gap=5를 적용해 학습 라벨과 검증 구간이 겹치지 않게 합니다.
splits = expanding_splits(
    n_samples=len(X_selected),
    n_folds=N_FOLDS,
    min_train=MIN_TRAIN_SIZE,
    horizon=VALID_SIZE,
    gap=LABEL_HORIZON,
    label_horizon=LABEL_HORIZON,
)

# 요청한 12개 폴드가 생성되었는지 확인합니다.
if len(splits) != N_FOLDS:
    raise RuntimeError(
        f"워크포워드 폴드가 {N_FOLDS}개가 아니라 "
        f"{len(splits)}개 생성되었습니다."
    )

# 각 폴드의 시간 순서와 gap을 검사합니다.
for fold_number, (train_index, valid_index) in enumerate(
    splits,
    start=1,
):
    actual_gap = (
        int(valid_index[0])
        - int(train_index[-1])
        - 1
    )

    if train_index[-1] >= valid_index[0]:
        raise RuntimeError(
            f"{fold_number}번 폴드에서 "
            "학습과 검증의 시간 순서가 잘못되었습니다."
        )

    if actual_gap != LABEL_HORIZON:
        raise RuntimeError(
            f"{fold_number}번 폴드의 gap이 "
            f"{actual_gap}개입니다."
        )

print("워크포워드 폴드 수:", len(splits))
print("최초 학습 데이터 수:", len(splits[0][0]))
print("폴드별 검증 데이터 수:", len(splits[0][1]))
print("학습·검증 사이 gap:", LABEL_HORIZON)

워크포워드 폴드 수: 12
최초 학습 데이터 수: 750
폴드별 검증 데이터 수: 60
학습·검증 사이 gap: 5


## 평가 지표

### Accuracy

전체 검증 데이터 중 모델이 하락·중립·상승을 정확하게 맞힌 비율이다. 특정 클래스에 예측이 치우칠 수 있으므로 Accuracy만 단독으로 사용하지 않는다.

### Macro F1-score

하락·중립·상승의 F1-score를 같은 비중으로 평균한 값이다. 특정 클래스의 예측을 포기한 모델을 확인하는 데 사용한다.

### Balanced Accuracy

하락·중립·상승의 Recall을 같은 비중으로 평균한 값이다. 클래스별 데이터 개수가 다를 때 일반 Accuracy를 보완한다.

### 방향 적중률

실제 중립 데이터를 제외하고 실제 하락과 상승을 얼마나 맞혔는지 계산한다.

### 기준선

항상 상승 예측과 학습 구간의 다수 클래스 예측을 기준선으로 사용한다. 모델이 단순한 예측보다 실제로 나은지 비교한다.

In [5]:
# 폴드별 평가 결과를 저장할 목록입니다.
fold_results = []

# 전체 OOS 검증 결과를 저장할 목록입니다.
all_y_true = []
all_y_pred = []
all_probabilities = []

# 기준선 예측 결과를 저장할 목록입니다.
all_always_up_pred = []
all_direction_majority_pred = []
all_three_class_majority_pred = []

# 12개 워크포워드 폴드를 시간순으로 학습합니다.
for fold_number, (train_index, valid_index) in enumerate(
    splits,
    start=1,
):
    # 현재 폴드의 학습·검증 데이터를 선택합니다.
    X_train = X_selected.iloc[train_index]
    X_valid = X_selected.iloc[valid_index]

    y_train = y_numeric[train_index]
    y_valid = y_numeric[valid_index]

    # 폴드마다 새로운 RandomForest을 생성합니다.
    #
    # 트리 기반 모델은 StandardScaler 없이 현재 폴드의 학습 데이터만 사용합니다.
    model = build_random_forest_baseline(class_weight="balanced")

    # 과거 학습 데이터만 사용해 모델을 학습합니다.
    # 수익률 워밍업 결측은 폴드 안에서만 제외해 검증 기간을 유지합니다.
    train_usable = np.isfinite(X_train.to_numpy(dtype=float)).all(axis=1)
    valid_usable = np.isfinite(X_valid.to_numpy(dtype=float)).all(axis=1)
    X_train = X_train.loc[train_usable]
    y_train = y_train[train_usable]
    X_valid = X_valid.loc[valid_usable]
    y_valid = y_valid[valid_usable]

    model.fit(X_train, y_train)

    # 미래 검증 구간의 클래스와 확률을 예측합니다.
    y_pred = model.predict(X_valid)
    probabilities = model.predict_proba(X_valid)

    # RandomForest이 세 클래스를 모두 학습했는지 확인합니다.
    classifier = model

    if not np.array_equal(
        classifier.classes_,
        np.array(CLASS_LABELS),
    ):
        raise RuntimeError(
            f"{fold_number}번 폴드의 학습 클래스가 "
            f"{classifier.classes_}입니다."
        )

    # 각 검증 행의 예측 확률 합이 1인지 확인합니다.
    if not np.allclose(
        probabilities.sum(axis=1),
        np.ones(len(X_valid)),
    ):
        raise RuntimeError(
            f"{fold_number}번 폴드의 예측 확률 합이 1이 아닙니다."
        )

    # 모든 데이터를 상승으로 예측하는 기준선을 만듭니다.
    always_up_pred = always_up(len(y_valid))

    # 학습 구간에서 상승·하락 중 더 많았던 방향을 예측합니다.
    direction_majority_pred = majority_class(
        y_train,
        len(y_valid),
    )

    # 중립을 포함한 세 클래스 중 가장 많았던 클래스를 찾습니다.
    train_labels, train_counts = np.unique(
        y_train,
        return_counts=True,
    )

    three_class_majority_label = int(
        train_labels[np.argmax(train_counts)]
    )

    three_class_majority_pred = np.full(
        shape=len(y_valid),
        fill_value=three_class_majority_label,
        dtype=int,
    )

    # 현재 폴드의 성능과 예측 개수를 저장합니다.
    fold_results.append(
        {
            "fold": fold_number,
            "train_size": len(train_index),
            "valid_size": len(valid_index),
            "train_end": df.iloc[train_index[-1]]["bas_dd"],
            "valid_start": df.iloc[valid_index[0]]["bas_dd"],
            "valid_end": df.iloc[valid_index[-1]]["bas_dd"],
            "accuracy": accuracy_score(
                y_valid,
                y_pred,
            ),
            "macro_f1": f1_score(
                y_valid,
                y_pred,
                labels=CLASS_LABELS,
                average="macro",
                zero_division=0,
            ),
            "balanced_accuracy": balanced_accuracy_score(
                y_valid,
                y_pred,
            ),
            "direction_hit_rate": hit_rate(
                y_pred,
                y_valid,
            ),
            "always_up_accuracy": accuracy_score(
                y_valid,
                always_up_pred,
            ),
            "three_class_majority_accuracy": accuracy_score(
                y_valid,
                three_class_majority_pred,
            ),
            "always_up_direction_hit_rate": hit_rate(
                always_up_pred,
                y_valid,
            ),
            "direction_majority_hit_rate": hit_rate(
                direction_majority_pred,
                y_valid,
            ),
            "predicted_down": int(np.sum(y_pred == -1)),
            "predicted_neutral": int(np.sum(y_pred == 0)),
            "predicted_up": int(np.sum(y_pred == 1)),
        }
    )

    # 전체 OOS 평가를 위해 현재 폴드 결과를 저장합니다.
    all_y_true.append(y_valid)
    all_y_pred.append(y_pred)
    all_probabilities.append(probabilities)

    all_always_up_pred.append(always_up_pred)
    all_direction_majority_pred.append(
        direction_majority_pred
    )
    all_three_class_majority_pred.append(
        three_class_majority_pred
    )

    print(
        f"{fold_number:02d}번 폴드 완료 | "
        f"학습 {len(train_index)}개 | "
        f"검증 {len(valid_index)}개 | "
        f"하락 {np.sum(y_pred == -1)}개 | "
        f"중립 {np.sum(y_pred == 0)}개 | "
        f"상승 {np.sum(y_pred == 1)}개"
    )

print("전체 워크포워드 학습이 완료되었습니다.")


01번 폴드 완료 | 학습 750개 | 검증 60개 | 하락 14개 | 중립 7개 | 상승 39개
02번 폴드 완료 | 학습 932개 | 검증 60개 | 하락 18개 | 중립 18개 | 상승 24개


03번 폴드 완료 | 학습 1114개 | 검증 60개 | 하락 14개 | 중립 17개 | 상승 29개
04번 폴드 완료 | 학습 1295개 | 검증 60개 | 하락 10개 | 중립 19개 | 상승 31개


05번 폴드 완료 | 학습 1477개 | 검증 60개 | 하락 18개 | 중립 25개 | 상승 17개
06번 폴드 완료 | 학습 1659개 | 검증 60개 | 하락 22개 | 중립 27개 | 상승 11개


07번 폴드 완료 | 학습 1841개 | 검증 60개 | 하락 6개 | 중립 40개 | 상승 14개
08번 폴드 완료 | 학습 2023개 | 검증 60개 | 하락 15개 | 중립 24개 | 상승 21개


09번 폴드 완료 | 학습 2205개 | 검증 60개 | 하락 17개 | 중립 26개 | 상승 17개
10번 폴드 완료 | 학습 2386개 | 검증 60개 | 하락 14개 | 중립 22개 | 상승 24개


11번 폴드 완료 | 학습 2568개 | 검증 60개 | 하락 19개 | 중립 22개 | 상승 19개
12번 폴드 완료 | 학습 2750개 | 검증 60개 | 하락 18개 | 중립 28개 | 상승 14개
전체 워크포워드 학습이 완료되었습니다.


In [6]:
# 폴드별 결과를 데이터프레임으로 변환합니다.
fold_results_df = pd.DataFrame(fold_results)

# 폴드별 기간, 성능과 예측 클래스 개수를 확인합니다.
display_columns = [
    "fold",
    "train_size",
    "valid_size",
    "train_end",
    "valid_start",
    "valid_end",
    "accuracy",
    "macro_f1",
    "balanced_accuracy",
    "direction_hit_rate",
    "predicted_down",
    "predicted_neutral",
    "predicted_up",
]

display(
    fold_results_df.loc[:, display_columns].round(4)
)

,fold,train_size,valid_size,train_end,valid_start,valid_end,accuracy,macro_f1,balanced_accuracy,direction_hit_rate,predicted_down,predicted_neutral,predicted_up
0,1,750,60,20130401,20130409,20130704,0.3333,0.2972,0.3672,0.4865,14,7,39
1,2,932,60,20131224,20140106,20140401,0.4167,0.3996,0.4389,0.5000,18,18,24
2,3,1114,60,20140924,20141002,20141229,0.2667,0.2741,0.2798,0.3125,14,17,29
3,4,1295,60,20150619,20150629,20150921,0.3500,0.3423,0.3803,0.3659,10,19,31
4,5,1477,60,20160316,20160324,20160621,0.3167,0.3042,0.3025,0.2857,18,25,17
5,6,1659,60,20161208,20161216,20170315,0.4000,0.3236,0.3333,0.2500,22,27,11
6,7,1841,60,20170904,20170912,20171212,0.3833,0.2662,0.2784,0.1071,6,40,14
7,8,2023,60,20180607,20180618,20180910,0.3833,0.3620,0.3658,0.3226,15,24,21
8,9,2205,60,20190308,20190318,20190612,0.4000,0.3831,0.3878,0.2973,17,26,17
9,10,2386,60,20191128,20191206,20200305,0.2500,0.2479,0.2665,0.1628,14,22,24


In [7]:
# 12개 폴드의 검증 결과를 하나의 배열로 합칩니다.
oos_y_true = np.concatenate(all_y_true)
oos_y_pred = np.concatenate(all_y_pred)
oos_probabilities = np.vstack(all_probabilities)

# 기준선 예측 결과도 하나의 배열로 합칩니다.
oos_always_up_pred = np.concatenate(
    all_always_up_pred
)

oos_direction_majority_pred = np.concatenate(
    all_direction_majority_pred
)

oos_three_class_majority_pred = np.concatenate(
    all_three_class_majority_pred
)

# 전체 OOS 결과를 요약합니다.
summary = pd.Series(
    {
        "사용 피처 수": len(FEATURE_COMBINATION),
        "워크포워드 폴드 수": len(fold_results_df),
        "전체 OOS 표본 수": len(oos_y_true),
        "평균 accuracy": fold_results_df[
            "accuracy"
        ].mean(),
        "accuracy 표준편차": fold_results_df[
            "accuracy"
        ].std(ddof=1),
        "평균 macro F1": fold_results_df[
            "macro_f1"
        ].mean(),
        "macro F1 표준편차": fold_results_df[
            "macro_f1"
        ].std(ddof=1),
        "전체 OOS accuracy": accuracy_score(
            oos_y_true,
            oos_y_pred,
        ),
        "전체 OOS macro F1": f1_score(
            oos_y_true,
            oos_y_pred,
            labels=CLASS_LABELS,
            average="macro",
            zero_division=0,
        ),
        "전체 OOS balanced accuracy": (
            balanced_accuracy_score(
                oos_y_true,
                oos_y_pred,
            )
        ),
        "전체 OOS 방향 적중률": hit_rate(
            oos_y_pred,
            oos_y_true,
        ),
        "항상 상승 accuracy": accuracy_score(
            oos_y_true,
            oos_always_up_pred,
        ),
        "3분류 다수 클래스 accuracy": accuracy_score(
            oos_y_true,
            oos_three_class_majority_pred,
        ),
        "항상 상승 방향 적중률": hit_rate(
            oos_always_up_pred,
            oos_y_true,
        ),
        "방향 다수 클래스 적중률": hit_rate(
            oos_direction_majority_pred,
            oos_y_true,
        ),
        "하락 예측 수": int(np.sum(oos_y_pred == -1)),
        "중립 예측 수": int(np.sum(oos_y_pred == 0)),
        "상승 예측 수": int(np.sum(oos_y_pred == 1)),
    },
    name="결과",
)

display(summary.to_frame().round(4))

,결과
사용 피처 수,7.0000
워크포워드 폴드 수,12.0000
전체 OOS 표본 수,720.0000
평균 accuracy,0.3542
accuracy 표준편차,0.0916
평균 macro F1,0.3233
macro F1 표준편차,0.0829
전체 OOS accuracy,0.3542
전체 OOS macro F1,0.3424
전체 OOS balanced accuracy,0.3430


In [8]:
# 실제 클래스별 데이터 개수를 계산합니다.
actual_counts = (
    pd.Series(oos_y_true)
    .value_counts()
    .reindex(CLASS_LABELS, fill_value=0)
)

# 예측 클래스별 데이터 개수를 계산합니다.
predicted_counts = (
    pd.Series(oos_y_pred)
    .value_counts()
    .reindex(CLASS_LABELS, fill_value=0)
)

# 실제값과 예측값 분포를 비교합니다.
prediction_distribution_df = pd.DataFrame(
    {
        "클래스": [
            f"{CLASS_NAMES[label]}({label})"
            for label in CLASS_LABELS
        ],
        "실제 개수": actual_counts.to_numpy(),
        "예측 개수": predicted_counts.to_numpy(),
    }
)

display(prediction_distribution_df)

,클래스,실제 개수,예측 개수
0,하락(-1),204,185
1,중립(0),306,275
2,상승(1),210,260


In [9]:
# 전체 OOS 데이터에서 클래스별 예측 확률을 확인합니다.
probability_columns = [
    f"{CLASS_NAMES[label]}({label})"
    for label in CLASS_LABELS
]

oos_probability_df = pd.DataFrame(
    oos_probabilities,
    columns=probability_columns,
)

# 각 클래스의 평균·최대 확률과 최종 선택 횟수를 계산합니다.
probability_summary = pd.DataFrame(
    {
        "평균 예측 확률": (
            oos_probability_df.mean()
        ),
        "최대 예측 확률": (
            oos_probability_df.max()
        ),
        "가장 높은 확률로 선택된 횟수": (
            oos_probability_df.idxmax(axis=1)
            .value_counts()
            .reindex(
                probability_columns,
                fill_value=0,
            )
        ),
    }
)

display(probability_summary.round(4))

,평균 예측 확률,최대 예측 확률,가장 높은 확률로 선택된 횟수
하락(-1),0.3007,0.80,185
중립(0),0.3573,0.78,275
상승(1),0.3420,0.79,260


In [10]:
# 혼동행렬에 표시할 클래스 이름을 지정합니다.
display_class_names = [
    "하락(-1)",
    "중립(0)",
    "상승(1)",
]

# 전체 OOS 예측 결과의 혼동행렬을 계산합니다.
confusion_matrix_df = pd.DataFrame(
    confusion_matrix(
        oos_y_true,
        oos_y_pred,
        labels=CLASS_LABELS,
    ),
    index=[
        f"실제 {name}"
        for name in display_class_names
    ],
    columns=[
        f"예측 {name}"
        for name in display_class_names
    ],
)

display(confusion_matrix_df)

,예측 하락(-1),예측 중립(0),예측 상승(1)
실제 하락(-1),53,75,76
실제 중립(0),66,129,111
실제 상승(1),66,71,73


In [11]:
# 이 셀만 따로 실행해도 작동하도록 클래스 이름을 다시 지정합니다.
report_class_names = [
    "하락(-1)",
    "중립(0)",
    "상승(1)",
]

# 하락·중립·상승의 Precision, Recall, F1-score를 계산합니다.
classification_report_df = pd.DataFrame(
    classification_report(
        oos_y_true,
        oos_y_pred,
        labels=CLASS_LABELS,
        target_names=report_class_names,
        output_dict=True,
        zero_division=0,
    )
).transpose()

display(classification_report_df.round(4))

,precision,recall,f1-score,support
하락(-1),0.2865,0.2598,0.2725,204.0000
중립(0),0.4691,0.4216,0.4441,306.0000
상승(1),0.2808,0.3476,0.3106,210.0000
accuracy,0.3542,0.3542,0.3542,0.3542
macro avg,0.3454,0.3430,0.3424,720.0000
weighted avg,0.3624,0.3542,0.3565,720.0000


In [12]:
# 실행 결과에서 해석 문장을 자동으로 만듭니다.
from IPython.display import Markdown

down_recall = float(classification_report_df.loc["하락(-1)", "recall"])
neutral_recall = float(classification_report_df.loc["중립(0)", "recall"])
up_recall = float(classification_report_df.loc["상승(1)", "recall"])
majority_baseline = 0.4250

display(Markdown(f"""
# RandomForest 조합 C 실험 결과 해석

- 클래스 가중치: `class_weight="balanced"`
- 전체 OOS Accuracy: `{summary['전체 OOS accuracy']:.4f}`
- 전체 OOS Macro F1: `{summary['전체 OOS macro F1']:.4f}`
- 전체 OOS balanced accuracy: `{summary['전체 OOS balanced accuracy']:.4f}`
- 하락 Recall: `{down_recall:.4f}`
- 중립 Recall: `{neutral_recall:.4f}`
- 상승 Recall: `{up_recall:.4f}`
- 예측 개수:
  - 하락 `{int(summary['하락 예측 수'])}`
  - 중립 `{int(summary['중립 예측 수'])}`
  - 상승 `{int(summary['상승 예측 수'])}`
- 최빈 클래스 Accuracy 기준선 대비: `{summary['전체 OOS accuracy'] - majority_baseline:+.4f}`

이 결과는 KOSPI200 개발구간의 동일한 12폴드·720개 검증 표본에서 측정했다.
최고 Accuracy가 최빈 클래스 기준선 `0.4250`을 넘지 못하면 최종 매매 신호로 채택할
근거가 없다. 네 모델의 최종 비교는 `05.모델비교.ipynb`에서 확인한다.
"""))



# RandomForest 조합 C 실험 결과 해석

- 클래스 가중치: `class_weight="balanced"`
- 전체 OOS Accuracy: `0.3542`
- 전체 OOS Macro F1: `0.3424`
- 전체 OOS balanced accuracy: `0.3430`
- 하락 Recall: `0.2598`
- 중립 Recall: `0.4216`
- 상승 Recall: `0.3476`
- 예측 개수:
  - 하락 `185`
  - 중립 `275`
  - 상승 `260`
- 최빈 클래스 Accuracy 기준선 대비: `-0.0708`

이 결과는 KOSPI200 개발구간의 동일한 12폴드·720개 검증 표본에서 측정했다.
최고 Accuracy가 최빈 클래스 기준선 `0.4250`을 넘지 못하면 최종 매매 신호로 채택할
근거가 없다. 네 모델의 최종 비교는 `05.모델비교.ipynb`에서 확인한다.
